In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler

In [16]:
mlflow.end_run()
mlflow.end_run()
mlflow.set_tracking_uri('http://localhost:5001')
mlflow.set_experiment("brecha_baseline_ols")

<Experiment: artifact_location=('/Users/santiagoquintana/Downloads/UNIANDES '
 'Local/M-IIND/2026-1/Analitic_comp/Proyecto2/ciencia_datos/ciencia_datosSQ/mlartifacts/3'), creation_time=1779739987540, experiment_id='3', last_update_time=1779739987540, lifecycle_stage='active', name='brecha_baseline_ols', tags={}, trace_location=None, workspace='default'>

In [17]:
data = pd.read_csv('../../EDA/data_finalSQ.csv')


print(f"\nVariable target (brecha_mat_lec):")
print(f"  Media: {data['brecha_mat_lec'].mean():.2f}")
print(f"  Desv: {data['brecha_mat_lec'].std():.2f}")
print(f"  Rango: [{data['brecha_mat_lec'].min():.2f}, {data['brecha_mat_lec'].max():.2f}]")

numeric_cols = data.select_dtypes(include=['number']).columns.tolist()
numeric_cols.remove('brecha_mat_lec')

X_train, X_test, y_train, y_test = train_test_split(data[numeric_cols], data['brecha_mat_lec'], test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_data = pd.DataFrame(X_train_scaled, columns=numeric_cols, index=X_train.index)
train_data['brecha_mat_lec'] = y_train.values
test_data = pd.DataFrame(X_test_scaled, columns=numeric_cols, index=X_test.index)
test_data['brecha_mat_lec'] = y_test.values


Variable target (brecha_mat_lec):
  Media: 0.78
  Desv: 8.17
  Rango: [-43.00, 51.00]


In [18]:
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('brecha_mat_lec')

correlations = data[numeric_cols + ['brecha_mat_lec']].corr()['brecha_mat_lec'].drop('brecha_mat_lec').sort_values(ascending=False)

print("\nTop 10 correlaciones (positivas y negativas):")
print(correlations.head(10))
print("\nCorrelaciones débiles (|r| < 0.1):")
weak_corr = correlations[correlations.abs() < 0.1]
print(f"  {len(weak_corr)} variables tienen correlación débil")


Top 10 correlaciones (positivas y negativas):
cole_jornada            0.071613
fami_educacionpadre     0.049105
fami_educacionmadre     0.043737
fami_personashogar      0.021280
fami_estratovivienda    0.018039
punt_tecnologia         0.017373
fami_cuartoshogar       0.005371
cole_area_ubicacion     0.004646
cole_caracter          -0.011518
cole_naturaleza        -0.013241
Name: brecha_mat_lec, dtype: float64

Correlaciones débiles (|r| < 0.1):
  10 variables tienen correlación débil


In [19]:
formula = "brecha_mat_lec ~ " + " + ".join(numeric_cols)

print(f"\nVariables a analizar: {len(numeric_cols)}")
print(f"Variables: {', '.join(numeric_cols[:8])}...")
print(f"\nAjustando modelo OLS...")

modelo_ols = smf.ols(formula, data=train_data).fit()
print(modelo_ols.summary())


Variables a analizar: 10
Variables: fami_estratovivienda, fami_educacionmadre, fami_educacionpadre, fami_cuartoshogar, fami_personashogar, cole_area_ubicacion, cole_caracter, cole_jornada...

Ajustando modelo OLS...
                            OLS Regression Results                            
Dep. Variable:         brecha_mat_lec   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     68.67
Date:                Mon, 25 May 2026   Prob (F-statistic):          2.04e-140
Time:                        18:11:23   Log-Likelihood:            -2.6250e+05
No. Observations:               74654   AIC:                         5.250e+05
Df Residuals:                   74643   BIC:                         5.251e+05
Df Model:                          10                                         
Covariance Type:            nonrobust                                   

In [20]:
y_pred_test = modelo_ols.get_prediction(test_data).summary_frame()
y_pred = y_pred_test['mean'].values

rmse = np.sqrt(mean_squared_error(y_test.values, y_pred))
mae = mean_absolute_error(y_test.values, y_pred)
r2 = r2_score(y_test.values, y_pred)
residuos = y_test.values - y_pred

print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  R²: {r2:.4f}")
print(f"  Sesgo: {np.mean(residuos):.4f}")
print(f"  Train R²: {modelo_ols.rsquared:.4f}")
print(f"  Adj R²: {modelo_ols.rsquared_adj:.4f}")

  RMSE: 8.0794
  MAE: 6.3371
  R²: 0.0086
  Sesgo: -0.0284
  Train R²: 0.0091
  Adj R²: 0.0090


In [21]:
coefs = modelo_ols.params
pvals = modelo_ols.pvalues
conf_int = modelo_ols.conf_int()

sig_vars = pvals[pvals < 0.05].drop('Intercept', errors='ignore')


print(f"\nCoeficientes significativos (p < 0.05):")
coef_df = pd.DataFrame({
    'variable': coefs.index,
    'coeficiente': coefs.values,
    'p-value': pvals.values,
    'CI_lower': conf_int[0].values,
    'CI_upper': conf_int[1].values
}).sort_values('p-value')

for idx, row in coef_df[coef_df['p-value'] < 0.05].iterrows():
    sig = "***" if row['p-value'] < 0.001 else "**" if row['p-value'] < 0.01 else "*"
    print(f"  {row['variable']:25} coef={row['coeficiente']:8.4f} {sig} (p={row['p-value']:.4f})")

print(f"\nVariables NO significativas (p >= 0.05): {len(coef_df[coef_df['p-value'] >= 0.05])-1}")


Coeficientes significativos (p < 0.05):
  Intercept                 coef=  0.7814 *** (p=0.0000)
  cole_jornada              coef=  0.5686 *** (p=0.0000)
  fami_personashogar        coef=  0.3058 *** (p=0.0000)
  fami_educacionpadre       coef=  0.3233 *** (p=0.0000)
  fami_educacionmadre       coef=  0.2582 *** (p=0.0000)
  cole_caracter             coef= -0.1181 *** (p=0.0004)
  fami_cuartoshogar         coef= -0.1150 *** (p=0.0009)
  fami_estratovivienda      coef= -0.0737 * (p=0.0437)

Variables NO significativas (p >= 0.05): 2


In [22]:
fig, ax = plt.subplots(figsize=(10, 6))
top_corr = correlations.abs().nlargest(15).sort_values()
colors = ['green' if correlations[var] > 0 else 'red' for var in top_corr.index]
ax.barh(range(len(top_corr)), top_corr.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=9)
ax.set_xlabel('Correlación con brecha_mat_lec (valor absoluto)', fontsize=10)
ax.set_title('Top 15 Variables por Correlación con Target', fontsize=11, fontweight='bold')
ax.axvline(0.1, color='gray', linestyle='--', alpha=0.5, label='Umbral correlación débil')
ax.legend()
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
mlflow.log_figure(fig, "05_ols_correlacion.png")
plt.close(fig)

In [23]:
fig, ax = plt.subplots(figsize=(10, 8))
coef_plot = coef_df[coef_df['variable'] != 'Intercept'].sort_values('coeficiente')
colors = ['red' if x < 0 else 'green' for x in coef_plot['coeficiente']]
alpha_vals = [0.9 if p < 0.05 else 0.4 for p in coef_plot['p-value']]

for i, (idx, row) in enumerate(coef_plot.iterrows()):
    ax.barh(i, row['coeficiente'], color=colors[i], alpha=alpha_vals[i])
    
ax.set_yticks(range(len(coef_plot)))
ax.set_yticklabels(coef_plot['variable'].values, fontsize=8)
ax.set_xlabel('Coeficiente OLS', fontsize=10)
ax.set_title('Coeficientes OLS (colores oscuros = p<0.05)', fontsize=11, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
mlflow.log_figure(fig, "06_ols_coeficientes.png")
plt.close(fig)

In [24]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test.values, y_pred, alpha=0.3, s=10, color='steelblue')
ax.plot([y_test.min(), y_test.max()], 
        [y_test.min(), y_test.max()], 
        'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Brecha real')
ax.set_ylabel('Brecha predicha (OLS)')
ax.set_title(f'OLS: Real vs Predicho (R²={r2:.4f})', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
mlflow.log_figure(fig, "07_ols_real_vs_pred.png")
plt.close(fig)

In [25]:
mlflow.end_run()

# Registrar en MLflow
with mlflow.start_run(run_name="OLS_all_variables"):
    mlflow.log_params({
        "modelo_tipo": "OLS",
        "n_features_total": len(numeric_cols),
        "n_features_significant": len(sig_vars),
    })
    
    mlflow.log_metrics({
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
        "train_r2": modelo_ols.rsquared,
        "adj_r2": modelo_ols.rsquared_adj,
        "f_statistic": modelo_ols.fvalue,
        "aic": modelo_ols.aic,
        "bic": modelo_ols.bic,
    })
    
    print(f"\n✓ Modelo OLS registrado en MLflow")
    print(f"  Run name: OLS_all_variables")
    print(f"  Variables significativas: {len(sig_vars)}/{len(numeric_cols)}")
    print(f"  Métricas: RMSE={rmse:.4f}, R²={r2:.4f}")

🏃 View run upset-koi-212 at: http://localhost:5001/#/experiments/3/runs/f1614f3be65b44e0a091d543cd455742
🧪 View experiment at: http://localhost:5001/#/experiments/3

✓ Modelo OLS registrado en MLflow
  Run name: OLS_all_variables
  Variables significativas: 7/10
  Métricas: RMSE=8.0794, R²=0.0086
🏃 View run OLS_all_variables at: http://localhost:5001/#/experiments/3/runs/e8a8f1acf8174ee085a8022d23900f85
🧪 View experiment at: http://localhost:5001/#/experiments/3


In [26]:
print(f"\nVariables SIGNIFICATIVAS (p < 0.05): {len(sig_vars)}/{len(numeric_cols)}")
print(f"\nVariables más importantes (por magnitud del coeficiente):")
top_coefs = coef_df[coef_df['variable'] != 'Intercept'].sort_values('coeficiente', key=abs, ascending=False).head(8)
for idx, row in top_coefs.iterrows():
    sig = "***" if row['p-value'] < 0.001 else "**" if row['p-value'] < 0.01 else "*" if row['p-value'] < 0.05 else "ns"
    print(f"  {row['variable']:25} coef={row['coeficiente']:8.4f} {sig}")

print(f"\nInterpretación del modelo:")
print(f"  F-statistic: {modelo_ols.fvalue:.2f} (p={modelo_ols.f_pvalue:.2e})")
print(f"  R² = {modelo_ols.rsquared:.4f} (explica el {modelo_ols.rsquared*100:.2f}% de varianza)")
print(f"  Adj R² = {modelo_ols.rsquared_adj:.4f} (penalizado por variables)")


Variables SIGNIFICATIVAS (p < 0.05): 7/10

Variables más importantes (por magnitud del coeficiente):
  cole_jornada              coef=  0.5686 ***
  fami_educacionpadre       coef=  0.3233 ***
  fami_personashogar        coef=  0.3058 ***
  fami_educacionmadre       coef=  0.2582 ***
  cole_caracter             coef= -0.1181 ***
  fami_cuartoshogar         coef= -0.1150 ***
  fami_estratovivienda      coef= -0.0737 *
  punt_tecnologia           coef=  0.0324 ns

Interpretación del modelo:
  F-statistic: 68.67 (p=2.04e-140)
  R² = 0.0091 (explica el 0.91% de varianza)
  Adj R² = 0.0090 (penalizado por variables)
